In [1]:
from jqdatasdk import *
import configparser
 
config = configparser.ConfigParser()
file = 'config.ini'
config.read(file)
user = config.get('login','user')
password = config.get('login','password')
auth(user,password)

auth success （JQData现有流量增加活动，详情请咨询JQData管理员，微信号：JQData02）


# 获取期货数据
## 获取期货概况
包含期货的开始时间、结束时间、代码、名称等。

更多的API可参考官方API文档

### 获取单只期货数据
获取单支期货的信息.

**调用方法**

get_security_info(code)

**参数**

code: 期货代码

**返回值**

一个对象, 有如下属性:

- display_name # 中文名称  
- name # 缩写简称  
- start_date # 开始日期, [datetime.date] 类型  
- end_date # 结束日期，[datetime.date] 类型  
- type # 类型，futures(期货)

**示例**

In [3]:
# 输出沪深300股指期货信息的最后交易日
get_security_info('IF1605.CCFX').end_date

datetime.date(2016, 5, 20)

### 获取所有期货数据
获取平台支持的所有股票、基金、指数、期货、期权信息

**参数**

- types: list: 用来过滤securities的类型, list元素可选: 'stock', 'fund', 'index', 'futures', 'options', 'etf', 'lof', 'fja', 'fjb', 'open_fund', 'bond_fund', 'stock_fund', 'QDII_fund', 'money_market_fund', 'mixture_fund'。 **types为空时返回所有股票, 不包括基金,指数和期货**
- date: 日期, 一个字符串或者 datetime.datetime /datetime.date 对象, 用于获取某日期还在上市的股票信息. 默认值为 None, 表示获取所有日期的股票信息.**建议使用时添加上指定date**

**返回** [pandas.DataFrame],

- display_name: 中文名称
- name: 缩写简称
- start_date: 上市日期
- end_date: 退市日期，如果没有退市则为2200-01-01
- type: 类型，stock(股票)，index(指数)，etf(ETF基金)，fja（分级A），fjb（分级B），fjm（分级母基金），mmf（场内交易的货币基金）open_fund（开放式基金）, bond_fund（债券基金）, stock_fund（股票型基金）, QDII_fund（QDII 基金）, money_market_fund（场外交易的货币基金）, mixture_fund（混合型基金）, options(期权)

**示例**

In [5]:
#获取2015-05-15在市的所有期货数据中的前5条
get_all_securities(['futures'],date='2019-05-15').head()

,display_name,name,start_date,end_date,type
A1905.XDCE,豆一1905,a1905,2017-11-15,2019-05-17,futures
A1907.XDCE,豆一1907,a1907,2018-01-16,2019-07-12,futures
A1909.XDCE,豆一1909,a1909,2018-03-15,2019-09-16,futures
A1911.XDCE,豆一1911,a1911,2018-05-16,2019-11-14,futures
A2001.XDCE,豆一2001,a2001,2018-07-16,2020-01-15,futures


### 期货结算价与持仓量
得到期货在一段时间的的如下的数据:

futures_sett_price: 期货结算价  
futures_positions: 期货持仓量(此数据将在行情数据中也进行提供,届时get_price/get_bars也可以获取到)

**调用方法**

get_extras(info, security_list, start_date=None, end_date=None, df=True):

**参数**

info: [‘futures_sett_price’，‘futures_positions’] 中的一个  
security_list: 标的列表  
start_date/end_date: 开始结束日期, 同[get_price]  
df: 返回[pandas.DataFrame]对象还是一个dict, 同[history]  

**返回值**

df=True:[pandas.DataFrame]对象, 列索引是期货代号, 行索引是[datetime.datetime]

**示例**

In [6]:
#查询'IF1604.CCFX', 'A9999.XDCE'的期货结算价，dataframe显示
get_extras('futures_sett_price', ['IF1604.CCFX', 'A9999.XDCE'], start_date='2016-04-06', end_date='2016-04-08',df=True)

,IF1604.CCFX,A9999.XDCE
2016-04-06,3235.4,3335.0
2016-04-07,3208.8,3329.0
2016-04-08,3164.0,3322.0


df=False 一个dict, key是基金代号, value是[numpy.ndarray]

**示例**

In [6]:
#查询'IF1604.CCFX', 'A9999.XDCE'的期货结算价，dict显示
get_extras('futures_sett_price', ['IF1604.CCFX', 'A9999.XDCE'], start_date='2016-04-06', end_date='2016-04-08',df=False)

{'IF1604.CCFX': array([3235.4, 3208.8, 3164. ]),
 'A9999.XDCE': array([3335., 3329., 3322.])}

## 获取期货行情数据
交易类数据提供期货的行情数据，通过API接口调用即可获取相应的数据。主要包括以下类别：

历史行情数据  
当前单位时间的行情数据  
### 行情数据
获取期货交易数据，可以通过参数设置获取日k线、分钟k线数据。获取数据的基本属性如下：

open 时间段开始时价格  
close 时间段结束时价格  
low 最低价  
high 最高价  
volume 成交的数量  
money 成交的金额
open_interest  持仓量(jqdatasdk即将上线此属性)
**调用方法**

get_price(security, start_date='2015-01-01', end_date='2015-12-31', frequency='daily', fields=None, skip_paused=False, fq='pre')  
**注**：
- (1)设定不同的unit参数，获取日K线或分钟k线，详情见参数。 这里请在使用时注意防止未来函数.  
- (2)security为列表时, 标的交易时间必须相同。

**参数**

security: 一支期货代码或者一个期货代码的list  
start_date: 字符串或者[datetime.datetime]/[datetime.date]对象, 开始时间, 默认是’2015-01-01’. 注意:
当取分钟数据时, 时间可以精确到分钟, 比如: 传入 datetime.datetime(2015, 1, 1, 10, 0, 0) 或者 '2015-01-01 10:00:00'.  
当取分钟数据时, 如果只传入日期, 则日内时间是当日的 00:00:00.  
当取天数据时, 传入的日内时间会被忽略  
- end_date: 格式同上, 结束时间, 默认是’2015-12-31’, 包含此日期. 注意: 当取分钟数据时, 如果 end_date 只有日期, 则日内时间等同于 00:00:00, 所以返回的数据是不包括 end_date 这一天的.
frequency: 单位时间长度, 几天或者几分钟, 现在支持’Xd’,’Xm’, ‘daily’(等同于’1d’), ‘minute’(等同于’1m’), X是一个正整数, 分别表示X天和X分钟(不论是按天还是按分钟回测都能拿到这两种单位的数据), 注意, 当X > 1时, field只支持[‘open’, ‘close’, ‘high’, ‘low’, ‘volume’, ‘money’]这几个标准字段. 默认值是daily
fields: 字符串list, 选择要获取的行情数据字段, 默认是None(表示[‘open’, ‘close’, ‘high’, ‘low’, ‘volume’, ‘money’]这几个标准字段), 支持属性里面的所有基本属性.  
skip_paused: 是否跳过停牌的时间, 如果不跳过, 停牌时会使用停牌前的数据填充(具体请看[SecurityUnitData]的paused属性), 但要注意:  
1. 默认为 False
2. 当 skip_paused 是 True 时, 只能取一只股票的信息
fq: 复权选项:  
1. 'pre': 前复权(根据’use_real_price’选项不同含义会有所不同, 参见set_option), 默认是前复权
2. None: 不复权, 返回实际价格
3. 'post': 后复权

**返回**

请注意, 为了方便比较一只期货的多个属性, 同时也满足对比多只期货的一个属性的需求, 我们在security参数是一只期货和多只期货时返回的结构完全不一样  
如果是一支期货, 则返回[pandas.DataFrame]对象, 行索引是[datetime.datetime]对象, 列索引是行情字段名字, 比如’open’/’close’. 比如: get_price('IC1505.CCFX', start_date='2015-05-06', end_date='2015-05-08')[:2] 返回:

In [8]:
get_price('IC1505.CCFX', start_date='2015-05-06', end_date='2015-05-08')[:2]

,open,close,high,low,volume,money
2015-05-06,8168.0,8040.8,8280.6,7970.0,111973.0,1.830955e+11
2015-05-07,8020.0,7911.4,8068.0,7901.0,109599.0,1.746042e+11


如果是多支期货, 则返回[pandas.Panel]对象, 里面是很多[pandas.DataFrame]对象, 索引是行情字段(open/close/…), 每个[pandas.DataFrame]的行索引是[datetime.datetime]对象, 列索引是期货代号. 比如get_price(['IC1505.CCFX', 'IC1506.CCFX'], start_date='2015-05-06', end_date='2015-05-08')['open'][:2]返回:

In [9]:
get_price(['IC1505.CCFX', 'IC1506.CCFX'], start_date='2015-05-06', end_date='2015-05-08')['open'][:2]

,IC1505.CCFX,IC1506.CCFX
2015-05-06,8168.0,8118.2
2015-05-07,8020.0,7900.0


**示例**

In [10]:
# 获取一支期货
df = get_price(['IC1505.CCFX', 'IC1506.CCFX'], start_date='2015-05-06', end_date='2015-05-08') # 获取IC1506.CCFX的2015年的按天数据

In [11]:
df = get_price('IC1506.CCFX', start_date='2015-05-06', end_date='2015-05-08', frequency='minute', fields=['open', 'close']) # 获得IC1506.CCFX的分钟数据, 只获取open+close字段

In [12]:
# 获取多只期货
panel =  get_price(['IC1505.CCFX', 'IC1506.CCFX'], start_date='2015-05-06', end_date='2015-05-08') # 获取中IC1505和IC1506的天数据, 返回一个[pandas.Panel]
df_open = panel['open']  # 获取开盘价的[pandas.DataFrame],  行索引是[datetime.datetime]对象, 列索引是期货代号
df_volume = panel['volume']  # 获取交易量的[pandas.DataFrame]
df_open['IC1506.CCFX'] # 获取IC1506的开盘价数据

2015-05-06    8118.2
2015-05-07    7900.0
2015-05-08    7894.8
Freq: D, Name: IC1506.CCFX, dtype: float64

### 获取期货tick数据
get_ticks(security, start_dt=None, end_dt=None, count=None, fields=None)
支持 2010-01-01 至今的盘后历史tick数据，提供买一卖一数据。（每0.5秒一次快照）如果要获取主力合约的tick数据，可以先使用get_dominant_future(underlying_symbol,dt)获取主力合约对应的标的，然后再用get_ticks()获取该合约的tick数据
Tick级回测模拟需要申请才能使用，申请链接

**参数：**

security: 股票代码或期货代码

start_dt: 开始日期

end_dt: 结束日期

count: 取出指定时间区间内前多少条的tick数据。

fields: 选择要获取的行情数据字段，默认为None，返回结果如下：

 

| 字段名   | 说明             | 字段类型 |
| -------- | ---------------- | -------- |
| datetime | 时间             | datetime |
| current  | 当前价           | float    |
| high     | 当日最高价       | float    |
| low      | 当日最低价       | float    |
| volume   | 累计成交量（手） | float    |
| money    | 累计成交额       | float    |
| position | 持仓量           | float    |
| a1_v     | 一档卖量         | float    |
| a1_p     | 一档卖价         | float    |
| b1_v     | 一档买量         | float    |
| b1_p     | 一档买价         | float    |

 

**期货tick数据示例：**

In [14]:
#获取AU1812期货合约在2018-07-02的tick数据,start_dt和count只能有一个不为None值
get_ticks('AU1812.XSGE',start_dt='2018-07-02',end_dt='2018-07-03',count=None).head()

,time,current,high,low,volume,money,position,a1_p,a1_v,b1_p,b1_v
0,2018-07-02 09:00:00.000,271.90,272.45,271.0,44536.0,1.210032e+10,365404.0,271.9,1.0,271.85,5.0
1,2018-07-02 09:00:01.000,271.85,272.45,271.0,44664.0,1.213512e+10,365338.0,271.9,2.0,271.85,10.0
2,2018-07-02 09:00:01.500,271.90,272.45,271.0,44674.0,1.213784e+10,365336.0,271.9,3.0,271.85,10.0
3,2018-07-02 09:00:02.000,272.00,272.45,271.0,44722.0,1.215089e+10,365300.0,272.0,191.0,271.95,4.0
4,2018-07-02 09:00:02.500,271.95,272.45,271.0,44748.0,1.215796e+10,365300.0,272.0,193.0,271.90,4.0


## 获取期货交易统计数据

### 获取期货龙虎榜数据
```
from jqdatasdk import finance  
finance.run_query(query(finance.FUT_MEMBER_POSITION_RANK).filter(finance.FUT_MEMBER_POSITION_RANK.code==code).limit(n))  
```
描述：记录各个期货交易所对不同商品下的期货合约，记录该交易所会员持仓排名前20的信息。（每天更新）

获取期货交易所会员持仓（龙虎榜）

**参数：**

query(finance.FUT_MEMBER_POSITION_RANK)：表示从finance.FUT_MEMBER_POSITION_RANK这张表中查询期货龙虎榜数据，还可以指定所要查询的字段名，格式如下：
- query(库名.表名.字段名1，库名.表名.字段名2），多个字段用逗号分隔进行提取；
- query函数的更多用法详见：[sqlalchemy.orm.query.Query对象](http://docs.sqlalchemy.org/en/rel_1_0/orm/query.html)    

finance.FUT_MEMBER_POSITION_RANK：收录了期货龙虎榜数据，表结构和字段信息如下：

**字段设计**

| 字段名称           | 中文名称               | 字段类型    | 含义                                                         |
| ------------------ | ---------------------- | ----------- | ------------------------------------------------------------ |
| day                | 交易日                 | date        |                                                              |
| code               | 合约编码               | varchar(12) | 同一商品根据交割日的不同对应不同的期货合约，比如：'CU1807.XSGE' |
| exchange           | 交易所编码             | varchar(10) | 英文编码 XSGE：上海期货交易所， XDCE：大连商品交易所， XZCE：郑州商品交易所， CCFX：中国金融期货交易所 |
| exchange_name      | 交易所名称             | varchar(30) |                                                              |
| underlying_code    | 标的编码               | varchar(10) |                                                              |
| underlying_name    | 标的名称               | varchar(50) |                                                              |
| rank_type_ID       | 排名类别编码           | int         | 501001-成交量排名, 501002-持买单量排名， 501003-持卖单量排名 |
| rank_type          | 排名类别               | varchar(50) | 包含:成交量排名，持买单量排名，持卖单量排名                  |
| rank               | 排名                   | int         |                                                              |
| member_name        | 会员简称               | varchar(50) |                                                              |
| indicator          | 统计指标               | int         | 统计指标根据排名类别确定，分别代表：成交量，持买单量，持卖单量。单位：手 |
| indicator_increase | 统计指标比上交易日增减 | int         | 单位：手                                                     |

**交易所编码**

| 交易所名称         | 编码 |
| ------------------ | ---- |
| 上海期货交易所     | XSGE |
| 大连商品交易所     | XDCE |
| 郑州商品交易所     | XZCE |
| 中国金融期货交易所 | CCFX |

**排名类别编码**

| 交易所名称   | 编码   |
| ------------ | ------ |
| 成交量排名   | 501001 |
| 持买单量排名 | 501002 |
| 持卖单量排名 | 501003 |

 

filter(finance.FUT_MEMBER_POSITION_RANK.code==code)：指定筛选条件，通过finance.FUT_MEMBER_POSITION_RANK.code==code可以指定你想要查询的合约编码；除此之外，还可以对表中其他字段指定筛选条件，如finance.FUT_MEMBER_POSITION_RANK.rank_type_ID=="501001"，表示按成交量排名查询龙虎榜数据；多个筛选条件用英文逗号分隔。  
limit(n)：限制返回的数据条数，n指定返回条数。

**返回结果：**

返回一个 dataframe，每一行对应数据表中的一条数据， 列索引是您所查询的字段名称

**注意：**

1. 为了防止返回数据量过大, 我们每次最多返回3000行
2. 不能进行连表查询，即同时查询多张表的数据

**示例：**

In [17]:
#查询A1905.XDCE最新一个交易日的持仓量龙虎榜数据
q=query(finance.FUT_MEMBER_POSITION_RANK.day,
        finance.FUT_MEMBER_POSITION_RANK.code,
        finance.FUT_MEMBER_POSITION_RANK.rank_type,
        finance.FUT_MEMBER_POSITION_RANK.rank,
        finance.FUT_MEMBER_POSITION_RANK.member_name,
finance.FUT_MEMBER_POSITION_RANK.indicator,).filter(finance.FUT_MEMBER_POSITION_RANK.code=='A1905.XDCE',finance.FUT_MEMBER_POSITION_RANK.rank_type_ID==501002).order_by(finance.FUT_MEMBER_POSITION_RANK.day.desc()).limit(20)
df=finance.run_query(q)
print(df.head())

          day        code rank_type  rank member_name  indicator
0  2019-04-26  A1905.XDCE    持买单量排名    20        国海良时        143
1  2019-04-26  A1905.XDCE    持买单量排名    19        大越期货        150
2  2019-04-26  A1905.XDCE    持买单量排名    18        广发期货        155
3  2019-04-26  A1905.XDCE    持买单量排名    17        国金期货        166
4  2019-04-26  A1905.XDCE    持买单量排名    16        中辉期货        172


### 获取期货仓单数据
```
from jqdatasdk import finance  
finance.run_query(query(finance.FUT_WAREHOUSE_RECEIPT).filter(finance.FUT_WAREHOUSE_RECEIPT.underlying_code==underlying_code).limit(n))  
```
描述：期货仓单是指由期货交易所指定交割仓库，按照期货交易所指定的程序，签发的符合合约规定质量的**实物提货凭证**。记录了交易所所有期货实物的库存情况以及变更情况。

**参数：**

- **query(finance.FUT_WAREHOUSE_RECEIPT)**：表示从finance.FUT_WAREHOUSE_RECEIPT这张表中查询期货仓单数据，还可以指定所要查询的字段名，格式如下：query(库名.表名.字段名1，库名.表名.字段名2），多个字段用逗号分隔进行提取；query函数的更多用法详见：[sqlalchemy.orm.query.Query对象](http://docs.sqlalchemy.org/en/rel_1_0/orm/query.html)
- **finance.FUT_WAREHOUSE_RECEIPT**：收录了期货仓单数据，表结构和字段信息如下：

**字段设计**

| 字段名称                          | 中文名称     | 字段类型    | 注释                                                         |
| --------------------------------- | ------------ | ----------- | ------------------------------------------------------------ |
| day                               | 日期         | date        |                                                              |
| exchange                          | 交易所编码   | varchar(10) | 英文编码                                                     |
| exchange_name                     | 交易所名称   | varchar(30) | 上海期货交易所，大连商品交易所，郑州商商品交易所，中国金融期货交易所 |
| underlying_code                   | 品种编码     | varchar(10) |                                                              |
| product_name                      | 品种名称     | varchar(20) |                                                              |
| warehouse_name                    | 仓库名称     | varchar(20) | 上期所：将地区和仓库数据合并成一条，仓库名称=“地区”+“仓库”。大商所：仓库名称存在多个不同的名字的，取第一个字体加粗的仓库名称。郑商所：不区分品牌，对每个仓库取仓库小计值 |
| warehouse_receipt_number          | 今日期货仓单 | int         |                                                              |
| unit                              | 单位         | varchar(10) |                                                              |
| warehouse_receipt_number_increase | 比昨日增减   | int         |                                                              |

**交易所名称编码**

| 交易所名称         | 编码 |
| ------------------ | ---- |
| 上海期货交易所     | XSGE |
| 大连商品交易所     | XDCE |
| 郑州商商品交易所   | XZCE |
| 中国金融期货交易所 | CCFX |

- **filter(finance.FUT_WAREHOUSE_RECEIPT.underlying_code==underlying_code)**：指定筛选条件，通过finance.FUT_WAREHOUSE_RECEIPT.underlying_code==underlying_code可以指定你想要查询的品种；除此之外，还可以对表中其他字段指定筛选条件；多个筛选条件用英文逗号分隔。
- **limit(n)**：限制返回的数据条数，n指定返回条数。

**返回结果：**

- 返回一个 dataframe，每一行对应数据表中的一条数据， 列索引是您所查询的字段名称

**注意：**

1. **为了防止返回数据量过大, 我们每次最多返回3000行**
2. 不能进行连表查询，即同时查询多张表的数据

**示例：**

In [18]:
#查询品种为铜（'CU')的2018年12月05日的仓单数据
q=query(finance.FUT_WAREHOUSE_RECEIPT.day,
        finance.FUT_WAREHOUSE_RECEIPT.underlying_code,
        finance.FUT_WAREHOUSE_RECEIPT.warehouse_name,
finance.FUT_WAREHOUSE_RECEIPT.warehouse_receipt_number).filter(finance.FUT_WAREHOUSE_RECEIPT.underlying_code=='CU',finance.FUT_WAREHOUSE_RECEIPT.day=='2018-12-05').order_by(finance.FUT_WAREHOUSE_RECEIPT.day.desc())
df=finance.run_query(q)
print(df.head())


          day underlying_code warehouse_name  warehouse_receipt_number
0  2018-12-05              CU        上海-上海裕强                         0
1  2018-12-05              CU        上海-上港物流                      3524
2  2018-12-05              CU       上海-世天威洋山                         0
3  2018-12-05              CU    上海-中储临港(保税)                         0
4  2018-12-05              CU        上海-中储吴淞                      5238


### 获取外盘期货日行情数据

```
from jqdatasdk import finance
df=finance.run_query(query(finance.FUT_GLOBAL_DAILY).filter(finance.FUT_GLOBAL_DAILY.day==date).limit(n))
```

描述：记录主要外盘商品期货的日行情数据，包含开盘价、收盘价、最高价、最低价、成交量等

**参数：**

- **query(finance.FUT_GLOBAL_DAILY)**：表示从finance.FUT_GLOBAL_DAILY这张表中查询外盘期货日行情数据，还可以指定所要查询的字段名，格式如下：query(库名.表名.字段名1，库名.表名.字段名2），多个字段用逗号分隔进行提取；query函数的更多用法详见：[query简易教程](https://www.joinquant.com/view/community/detail/16411)
- **finance.FUT_GLOBAL_DAILY**：收录了外盘期货日行情数据，表结构和字段信息如下：

**字段设计**

| **字段**   | **名称**    | **类型**      | **非空** | **含义**                                      |
| ---------- | ----------- | ------------- | -------- | --------------------------------------------- |
| code       | 期货代码    | varchar(64)   | Y        | 代码列表详见下方期货代码名称对照表            |
| name       | 期货名称    | varchar(64)   |          |                                               |
| day        | 日期        | date          | Y        |                                               |
| open       | 开盘价      | decimal(20,6) |          |                                               |
| close      | 收盘价      | decimal(20,6) |          |                                               |
| low        | 最低价      | decimal(20,6) |          |                                               |
| high       | 最高价      | decimal(20,6) |          |                                               |
| volume     | 成交量      | decimal(20,6) |          |                                               |
| change_pct | 涨跌幅（%） | decimal(20,4) |          | （当日收盘价-前收价）/前收价                  |
| amplitude  | 振幅（%）   | decimal(20,6) |          | （当日最高点的价格－当日最低点的价格）/前收价 |
| pre_close  | 前收价      | decimal(20,6) |          |                                               |

**期货代码名称对照表**

| 代码 | 名称         |
| ---- | ------------ |
| AHD  | LME铝3个月   |
| BO   | CBOT-黄豆油  |
| BP   | IMM-英镑     |
| C    | CBOT-玉米    |
| CAD  | LME铜3个月   |
| CD   | IMM-加元     |
| CL   | NYMEX原油    |
| DXF  | 美元指数期货 |
| EC   | IMM-欧元     |
| ES   | 标普期货     |
| GC   | COMEX黄金    |
| HG   | COMEX铜      |
| JY   | IMM-日元     |
| LHC  | CME-瘦肉猪   |
| NG   | NYMEX天然气  |
| NID  | LME镍3个月   |
| OIL  | 布伦特原油   |
| PBD  | LME铅3个月   |
| S    | CBOT-黄豆    |
| SF   | IMM-瑞郎     |
| SI   | COMEX白银    |
| SM   | CBOT-黄豆粉  |
| SND  | LME锡3个月   |
| W    | CBOT-小麦    |
| XAG  | 伦敦银       |
| XAU  | 伦敦金       |
| XPD  | 伦敦钯金     |
| XPT  | 伦敦铂金     |
| ZSD  | LME锌3个月   |

- **filter(finance.FUT_GLOBAL_DAILY.day==date)**：指定筛选条件，通过finance.FUT_GLOBAL_DAILY.day==date可以指定你想要查询的行情日期；除此之外，还可以对表中其他字段指定筛选条件，如finance.FUT_GLOBAL_DAILY.code==code，表示指定标的代码查询日行情；多个筛选条件用英文逗号分隔。
- **limit(n)**：限制返回的数据条数，n指定返回条数。

**返回结果：**

- 返回一个 dataframe，每一行对应数据表中的一条数据， 列索引是您所查询的字段名称

**注意：**

1. **为了防止返回数据量过大, 我们每次最多返回3000行**
2. 不能进行连表查询，即同时查询多张表的数据

**示例：**

In [17]:
#查询2019年6月27的外盘期货日行情数据
# from jqdatasdk import finance
df=finance.run_query(query(finance.FUT_GLOBAL_DAILY).filter(finance.FUT_GLOBAL_DAILY.day=='2019-06-27'))
df.head()

,id,code,name,day,open,close,low,high,volume,change_pct,amplitude,pre_close
0,93431,AHD,LME铝3个月,2019-06-27,1815.000,1797.000,1788.000,1825.000,11737.0,-0.9372,2.039691,1814.000
1,92994,BO,CBOT-黄豆油,2019-06-27,27.860,27.791,27.650,28.040,563.0,-0.1437,1.401315,27.831
2,93032,BP,IMM-英镑,2019-06-27,1.274,1.271,1.271,1.277,65982.0,-0.1571,0.471328,1.273
3,93127,C,CBOT-玉米,2019-06-27,442.750,440.001,438.500,445.500,1143.0,-0.6772,1.580132,443.001
4,93070,CAD,LME铜3个月,2019-06-27,5986.500,5998.500,5960.000,6016.000,12717.0,0.0834,0.934346,5993.500


# 期货列表
**注:** 支持的列表以及代码也可以使用以下代码进行筛选/获取

In [20]:
df = get_all_securities(types=['futures'],date=None)                       #获取期货合约
# dominant_and_idnex = df[df.end_date==datetime.date(2030,1,1)]  #商品指数和主力合约
dominant_future = [d for d in df.index if '9999' in d]           #所有品种的主力合约
index_future = [i for i in df.index if '8888' in i]              #所有品种的指数代码
code_name = [c[:-9] for c in index_future]                       #获取各个品种代号,使用get_dominant_future时会用到

## 金融期货列表
提供当前时间段内有效的金融期货合约数据（如行情数据等），具体调用方法可以参考API文档.(温馨提示：鉴于内容太多, 可使用Ctrl + F进行搜索)可使用get_all_securities(['futures'])获取当前阶段的金融期货列表包含期货代码、名称，起止日期等信息。或登录中金所官网进行查询。

## 支持的金融期货种类

| 代码        | 名称            | 代码        | 名称               |
| ----------- | --------------- | ----------- | ------------------ |
| IC9999.CCFX | 中证500主力合约 | T9999.CCFX  | 10年期国债主力合约 |
| IF9999.CCFX | 沪深300主力合约 | TF9999.CCFX | 5年期国债主力合约  |
| IH9999.CCFX | 上证50主力合约  | TS9999.CCFX | 2年期国债主力合约  |

#### 1. 沪深300指数期货合约表

| 信息                 | 内容                                         |
| -------------------- | -------------------------------------------- |
| 合约标的             | 沪深300指数                                  |
| 合约乘数             | 每点300元                                    |
| 报价单位             | 指数点                                       |
| 最小变动价位         | 0.2点                                        |
| 合约月份             | 当月、下月及随后两个季月                     |
| 交易时间             | 上午：9:30-11:30，下午：13:00-15:00          |
| 每日价格最大波动限制 | 上一个交易日结算价的±10%                     |
| 最低交易保证金       | 合约价值的8%                                 |
| 最后交易日           | 合约到期月份的第三个周五，遇国家法定假日顺延 |
| 交割日期             | 同最后交易日                                 |
| 交割方式             | 现金交割                                     |
| 交易代码             | IF                                           |
| 上市交易所           | 中国金融期货交易所                           |

#### 2. 上证50股指期货合约表

| 信息                 | 内容                                         |
| -------------------- | -------------------------------------------- |
| 合约标的             | 上证50指数                                   |
| 合约乘数             | 每点300元                                    |
| 报价单位             | 指数点                                       |
| 最小变动价位         | 0.2点                                        |
| 合约月份             | 当月、下月及随后两个季月                     |
| 交易时间             | 上午： 9:30-11:30，下午：13:00-15:00         |
| 每日价格最大波动限制 | 上一个交易日结算价的±10%                     |
| 最低交易保证金       | 合约价值的8%                                 |
| 最后交易日           | 合约到期月份的第三个周五，遇国家法定假日顺延 |
| 交割日期             | 同最后交易日                                 |
| 交割方式             | 现金交割                                     |
| 交易代码             | IH                                           |
| 上市交易所           | 中国金融期货交易所                           |


#### 3. 中证500股指期货合约表

| 信息                 | 内容                                         |
| -------------------- | -------------------------------------------- |
| 合约标的             | 中证500指数                                  |
| 合约乘数             | 每点200元                                    |
| 报价单位             | 指数点                                       |
| 最小变动价位         | 0.2点                                        |
| 合约月份             | 当月、下月及随后两个季月                     |
| 交易时间             | 上午： 9:30-11:30, 下午：13:00-15:00         |
| 每日价格最大波动限制 | 上一个交易日结算价的±10%                     |
| 最低交易保证金       | 合约价值的8%                                 |
| 最后交易日           | 合约到期月份的第三个周五，遇国家法定假日顺延 |
| 交割日期             | 同最后交易日                                 |
| 交割方式             | 现金交割                                     |
| 交易代码             | IC                                           |
| 上市交易所           | 中国金融期货交易所                           |


#### 4. 2年期国债期货合约表

| 信息                 | 内容                                                         |
| -------------------- | ------------------------------------------------------------ |
| 合约标的             | 面值为200万元人民币、票面利率为3%的名义中短期国债            |
| 可交割国债           | 发行期限不高于5年，合约到期月份首日剩余期限为1.5-2.25年的记账式附息国债 |
| 报价方式             | 百元净价报价                                                 |
| 最小变动价位         | 0.005元                                                      |
| 合约月份             | 最近的三个季月（3月、6月、9月、12月中的最近三个月循环）      |
| 交易时间             | 9:15 - 11:30, 13:00 - 15:15                                  |
| 最后交易日交易时间   | 9:15 - 11:30                                                 |
| 每日价格最大波动限制 | 上一交易日结算价的±0.5%                                      |
| 最低交易保证金       | 合约价值的0.5%                                               |
| 最后交易日           | 合约到期月份的第二个星期五                                   |
| 最后交割日           | 最后交易日后的第三个交易日                                   |
| 交割方式             | 实物交割                                                     |
| 交易代码             | TS                                                           |
| 上市交易所           | 中国金融期货交易所                                           |


#### 5. 5年期国债期货合约表

| 信息                 | 内容                                                    |
| -------------------- | ------------------------------------------------------- |
| 合约标的             | 面值为100万元人民币、票面利率为3%的名义中期国债         |
| 可交割国债           | 合约到期月份首日剩余期限为4-5.25年的记账式附息国债      |
| 报价方式             | 百元净价报价                                            |
| 最小变动价位         | 0.005元                                                 |
| 合约月份             | 最近的三个季月（3月、6月、9月、12月中的最近三个月循环） |
| 交易时间             | 09:15—11:30， 13:00—15:15                               |
| 最后交易日交易时间   | 09:15—11:30                                             |
| 每日价格最大波动限制 | 上一交易日结算价的±1.2%                                 |
| 最低交易保证金       | 合约价值的1%                                            |
| 最后交易日           | 合约到期月份的第二个星期五                              |
| 最后交割日           | 最后交易日后的第三个交易日                              |
| 交割方式             | 实物交割                                                |
| 交易代码             | TF                                                      |
| 上市交易所           | 中国金融期货交易所                                      |


#### 6. 10年期国债期货合约表

| 信息                 | 内容                                                    |
| -------------------- | ------------------------------------------------------- |
| 合约标的             | 面值为100万元人民币、票面利率为3%的名义长期国债         |
| 可交割国债           | 合约到期月份首日剩余期限为6.5-10.25年的记账式附息国债   |
| 报价方式             | 百元净价报价                                            |
| 最小变动价位         | 0.005元                                                 |
| 合约月份             | 最近的三个季月（3月、6月、9月、12月中的最近三个月循环） |
| 交易时间             | 9:15 - 11:30，13:00 - 15:15                             |
| 最后交易日交易时间   | 9:15 - 11:30                                            |
| 每日价格最大波动限制 | 上一交易日结算价的±2%                                   |
| 最低交易保证金       | 合约价值的2%                                            |
| 最后交易日           | 合约到期月份的第二个星期五                              |
| 最后交割日           | 最后交易日后的第三个交易日                              |
| 交割方式             | 实物交割                                                |
| 交易代码             | T                                                       |
| 上市交易所           | 中国金融期货交易所                                      |

## 商品期货列表

提供当前时间段内有效的商品期货期货合约数据（如行情数据等），具体调用方法可以参考API文档。(温馨提示：鉴于内容太多, 可使用Ctrl + F进行搜索)

指数合约的合成规则：使用该品种内所有正在交易的合约， 按照持仓额加权计算。


### 上海国际能源交易中心

| 名称     | 主力合约代码 | 指数合约代码 |
| -------- | ------------ | ------------ |
| 原油合约 | SC9999.XINE  | SC8888.XINE  |


### 上期所

| 名称         | 主力合约代码 | 指数合约代码 |
| ------------ | ------------ | ------------ |
| 白银合约     | AG9999.XSGE  | AG8888.XSGE  |
| 铝合约       | AL9999.XSGE  | AL8888.XSGE  |
| 黄金合约     | AU9999.XSGE  | AU8888.XSGE  |
| 沥青合约     | BU9999.XSGE  | BU8888.XSGE  |
| 铜合约       | CU9999.XSGE  | CU8888.XSGE  |
| 燃料油合约   | FU9999.XSGE  | FU8888.XSGE  |
| 热轧卷板合约 | HC9999.XSGE  | HC8888.XSGE  |
| 镍合约       | NI9999.XSGE  | NI8888.XSGE  |
| 铅合约       | PB9999.XSGE  | PB8888.XSGE  |
| 螺纹钢合约   | RB9999.XSGE  | RB8888.XSGE  |
| 天然橡胶合约 | RU9999.XSGE  | RU8888.XSGE  |
| 锡合约       | SN9999.XSGE  | SN8888.XSGE  |
| 线材合约     | WR9999.XSGE  | WR8888.XSGE  |
| 锌合约       | ZN9999.XSGE  | ZN8888.XSGE  |
| 纸浆合约     | SP9999.XSGE  | SP8888.XSGE  |


### 郑商所

| 名称         | 主力合约代码 | 指数合约代码 | 备注                                   |
| ------------ | ------------ | ------------ | -------------------------------------- |
| 苹果合约     | AP9999.XZCE  | AP8888.XZCE  |                                        |
| 棉花合约     | CF9999.XZCE  | CF8888.XZCE  |                                        |
| 棉纱合约     | CY9999.XZCE  | CY8888.XZCE  |                                        |
| 早籼稻合约   | ER9999.XZCE  | ER8888.XZCE  |                                        |
| 玻璃合约     | FG9999.XZCE  | FG8888.XZCE  |                                        |
| 绿豆合约     | GN9999.XZCE  | GN8888.XZCE  | 已于2009年5月5日退市                   |
| 粳稻谷合约   | JR9999.XZCE  | JR8888.XZCE  |                                        |
| 晚籼稻合约   | LR9999.XZCE  | LR8888.XZCE  |                                        |
| 甲醇合约     | MA9999.XZCE  | MA8888.XZCE  | MA为新的甲醇合约代码，自MA1506开始执行 |
| 甲醇合约     | ME9999.XZCE  | ME8888.XZCE  | ME为旧的甲醇合约代码，自ME1505停止执行 |
| 菜籽油合约   | OI9999.XZCE  | OI8888.XZCE  |                                        |
| 普麦合约     | PM9999.XZCE  | PM8888.XZCE  |                                        |
| 早籼稻合约   | RI9999.XZCE  | RI8888.XZCE  |                                        |
| 菜籽粕合约   | RM9999.XZCE  | RM8888.XZCE  |                                        |
| 菜籽油合约   | RO9999.XZCE  | RO8888.XZCE  |                                        |
| 油菜籽合约   | RS9999.XZCE  | RS8888.XZCE  |                                        |
| 硅铁合约     | SF9999.XZCE  | SF8888.XZCE  |                                        |
| 锰硅合约     | SM9999.XZCE  | SM8888.XZCE  |                                        |
| 白糖合约     | SR9999.XZCE  | SR8888.XZCE  |                                        |
| PTA合约      | TA9999.XZCE  | TA8888.XZCE  |                                        |
| 动力煤合约   | TC9999.XZCE  | TC8888.XZCE  |                                        |
| 强麦合约     | WH9999.XZCE  | WH8888.XZCE  | WH为新的强麦合约代码，自WH1307开始执行 |
| 强麦合约     | WS9999.XZCE  | WS8888.XZCE  | WS为旧的强麦合约代码，自WS1305停止执行 |
| 硬白小麦合约 | WT9999.XZCE  | WT8888.XZCE  |                                        |
| 动力煤合约   | ZC9999.XZCE  | ZC8888.XZCE  |                                        |
| 红枣合约     | CJ9999.XZCE  | CJ8888.XZCE  |                                        |

### 大商所

| 名称         | 主力合约代码 | 指数合约代码 |
| ------------ | ------------ | ------------ |
| 豆一合约     | A9999.XDCE   | A8888.XDCE   |
| 豆二合约     | B9999.XDCE   | B8888.XDCE   |
| 胶合板合约   | BB9999.XDCE  | BB8888.XDCE  |
| 玉米合约     | C9999.XDCE   | C8888.XDCE   |
| 玉米淀粉合约 | CS9999.XDCE  | CS8888.XDCE  |
| 纤维板合约   | FB9999.XDCE  | FB8888.XDCE  |
| 铁矿石合约   | I9999.XDCE   | I8888.XDCE   |
| 焦炭合约     | J9999.XDCE   | J8888.XDCE   |
| 鸡蛋合约     | JD9999.XDCE  | JD8888.XDCE  |
| 焦煤合约     | JM9999.XDCE  | JM8888.XDCE  |
| 聚乙烯合约   | L9999.XDCE   | L8888.XDCE   |
| 豆粕合约     | M9999.XDCE   | M8888.XDCE   |
| 棕榈油合约   | P9999.XDCE   | P8888.XDCE   |
| 聚丙烯合约   | PP9999.XDCE  | PP8888.XDCE  |
| 聚氯乙烯合约 | V9999.XDCE   | V8888.XDCE   |
| 豆油合约     | Y9999.XDCE   | Y8888.XDCE   |
| 乙二醇合约   | EG9999.XDCE  | EG8888.XDCE  |